# 01. Data Preparation for Reliable Feedback Reporting

This notebook documents how the feedback dataset was prepared before downstream analysis.

The purpose of this stage is to define a consistent analysis-ready population so that later comparisons are based on comparable records rather than differences in data quality.

> **Data note:** The original university feedback is not published. `sample_feedback.csv` contains synthetic examples only.


## Define analysis-ready feedback

The final workflow removed missing or blank comments, duplicate responses, and comments shorter than three words. Text was standardized while preserving sentence context needed for later text classification.

These rules help prevent double-counting, exclude records with too little information to interpret, and keep semester-level comparisons based on a consistent population. The final analytical dataset contained **1,022 comments** from an original 1,178 responses.


In [ ]:
from pathlib import Path
import re
import pandas as pd

DATA_PATH = Path('../data/sample_feedback.csv')
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
# Exclude records with no usable feedback
df_cleaned = df.dropna(subset=['comment']).copy()
df_cleaned = df_cleaned[df_cleaned['comment'].str.strip().ne('')]

# Avoid double-counting repeated feedback
df_cleaned = df_cleaned.drop_duplicates(subset=['comment']).copy()

# Remove responses too short to provide interpretable evidence
df_cleaned['word_count'] = df_cleaned['comment'].str.split().str.len()
df_cleaned = df_cleaned[df_cleaned['word_count'] >= 3].copy()


In [ ]:
def preprocess_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'[^\w\s.,!?-]', ' ', text)
    return ' '.join(text.split())

df_cleaned['comment_preprocessed'] = df_cleaned['comment'].apply(preprocess_text)
df_cleaned[['comment', 'comment_preprocessed']].head()

## Data-quality validation from the dissertation

The original research reported:

- Original responses: **1,178**
- Final analysis-ready responses: **1,022**
- Data retention rate: **86.76%**
- Missing/blank responses removed: **9**
- Duplicate comments removed: **81**
- Short responses (<3 words) removed: **147**

These figures provide a clear bridge between raw input and the population used for reporting. The synthetic sample below demonstrates the same checks but is not expected to reproduce the dissertation counts.


In [ ]:
# Basic quality checks for the public sample
quality_summary = {
    'rows': len(df_cleaned),
    'unique_comments': df_cleaned['comment_preprocessed'].nunique(),
    'avg_words_per_comment': round(df_cleaned['word_count'].mean(), 2),
}
quality_summary

## Reporting implication

The main output of this stage is not the cleaned file itself, but a transparent set of inclusion rules. Keeping those rules stable makes later semester comparisons easier to interpret and reduces the risk that changes in reported sentiment are caused by inconsistent preprocessing rather than genuine changes in student feedback.
